<a href="https://colab.research.google.com/github/Rds1007/SQL_BigDataInterview/blob/main/Best_Selling_Item.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Find the best-selling item for each month (no need to separate months by year). The best-selling item is determined by the highest total sales amount, calculated as: total_paid = unitprice * quantity. A negative quantity indicates a return or cancellation (the invoice number begins with 'C'. To calculate sales, ignore returns and cancellations. Output the month, description of the item, and the total amount paid.

country:
text
customerid:
double precision
description:
text
invoicedate:
date
invoiceno:
text
quantity:
bigint
stockcode:
text
unitprice:
double precision

https://platform.stratascratch.com/coding/10172-best-selling-item?code_type=**1**

In [9]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Initialize SparkSession
spark = SparkSession.builder.appName("BestSellingItem").getOrCreate()

print("SparkSession created successfully.")

SparkSession created successfully.


Since the `online_retail` table is not provided, I'll create a sample PySpark DataFrame with similar columns as implied by your SQL query. You would replace this with your actual data loading code (e.g., `spark.read.csv`, `spark.read.jdbc`, etc.).

In [10]:
online_retail_df=spark.read.csv('/content/sample_data/online_retail_sample.csv',header=True,inferSchema=True)

In [16]:
online_retail_df.show(10)

+---------+---------+--------------------+--------+-----------+---------+----------+--------------+
|invoiceno|stockcode|         description|quantity|invoicedate|unitprice|customerid|       country|
+---------+---------+--------------------+--------+-----------+---------+----------+--------------+
|   544586|    21890|S/6 WOODEN SKITTL...|       3| 2011-02-21|     2.95|     17338|United Kingdom|
|   541104|   84509G|SET OF 4 FAIRY CA...|       3| 2011-01-13|     3.29|      NULL|United Kingdom|
|   560772|    22499|WOODEN UNION JACK...|       3| 2011-07-20|     4.96|      NULL|United Kingdom|
|   555150|    22488|NATURAL SLATE REC...|       5| 2011-05-31|     3.29|      NULL|United Kingdom|
|   570521|    21625|VINTAGE UNION JAC...|       3| 2011-10-11|     6.95|     12371|   Switzerland|
|   547053|    22087|PAPER BUNTING WHI...|      40| 2011-03-20|     2.55|     13001|United Kingdom|
|   573360|    22591|CARDHOLDER GINGHA...|       6| 2011-10-30|     3.25|     15748|United Kingdom|


Now, let's apply the PySpark logic to find the best-selling item by month, mirroring your SQL query.

In [20]:
# Equivalent of CTE 'cte'
cte_df = online_retail_df.filter(~F.col("invoiceno").startswith('C')) \
                          .withColumn("month", F.month(F.col("invoicedate"))) \
                          .withColumn("total_paid", F.col("unitprice") * F.col("quantity"))
# Equivalent of CTE 'dte'
dte_df = cte_df.groupBy("month", "stockcode", "description") \
               .agg(F.sum("total_paid").alias("total_paid"))

# Define a window specification for ranking
window_spec = Window.partitionBy("month").orderBy(F.desc("total_paid"))

# Equivalent of CTE 'ete'
ete_df = dte_df.withColumn("ranks", F.rank().over(window_spec))

# Final selection and ordering
best_selling_items_df = ete_df.filter(F.col("ranks") == 1) \
                                  .select("month", "description", "total_paid") \
                                  .orderBy("month")



In [22]:
best_selling_items_df.show(15)

+-----+--------------------+------------------+
|month|         description|        total_paid|
+-----+--------------------+------------------+
|    1|LUNCH BAG SPACEBO...|             74.26|
|    2|REGENCY CAKESTAND...|             38.25|
|    3|PAPER BUNTING WHI...|             102.0|
|    4|  SPACEBOY LUNCH BOX|              23.4|
|    5|PAPER BUNTING WHI...|              51.0|
|    6|Dotcomgiftshop Gi...|             41.67|
|    7|PAPER BUNTING WHI...|56.099999999999994|
|    8|LUNCH BAG PINK PO...|              16.5|
|    9|RED RETROSPOT PEG...|             34.72|
|   10|CHOCOLATE HOT WAT...|             102.0|
|   11|RED WOOLLY HOTTIE...|228.25000000000003|
|   12|PAPER BUNTING RET...|35.400000000000006|
+-----+--------------------+------------------+



In [23]:
# Stop SparkSession
spark.stop()
print("SparkSession stopped.")

SparkSession stopped.
